In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir('/content/quantify_superadditivity'):
    !git clone https://github.com/angelafoua/quantify_superadditivity /content/quantify_superadditivity
%cd /content/quantify_superadditivity
!git checkout main && git pull origin main

In [ ]:
!pip install -q hydra-core==1.3.2 omegaconf==2.3.0 h5py networkx statsmodels seaborn tabulate tqdm scikit-learn pandas scipy medmnist
!pip install -q -e . --no-deps
!python scripts/validate_setup.py

In [ ]:
import os
DRIVE_OUT = '/content/drive/MyDrive/superadditivity_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
if os.path.islink('outputs') or os.path.exists('outputs'):
    !rm -rf outputs
os.symlink(DRIVE_OUT, 'outputs')
print('outputs ->', os.path.realpath('outputs'))

In [ ]:
# --- Cache datasets on Google Drive (same as CIFAR notebook) ---
import os
DRIVE_DATA = '/content/drive/MyDrive/superadditivity_data'
LOCAL_DATA = '/content/quantify_superadditivity/data'
os.makedirs(DRIVE_DATA, exist_ok=True)

if os.path.isdir(LOCAL_DATA) and not os.path.islink(LOCAL_DATA):
    !cp -rn {LOCAL_DATA}/* {DRIVE_DATA}/ 2>/dev/null || true
    !rm -rf {LOCAL_DATA}
if os.path.islink(LOCAL_DATA):
    os.remove(LOCAL_DATA)
os.symlink(DRIVE_DATA, LOCAL_DATA)
print('data ->', os.path.realpath(LOCAL_DATA))


In [ ]:
import os
import sys
sys.path.insert(0, '/content/quantify_superadditivity')

from superadditivity.datasets.dataset_loader import DatasetLoader

print("\n" + "="*70)
print("VALIDATING PathMNIST Dataset")
print("="*70)

# Create output dir for sample images
validation_output = os.path.join(DRIVE_OUT, 'pathmnist_validation')
os.makedirs(validation_output, exist_ok=True)

# Load and validate
try:
    loader = DatasetLoader(
        dataset_name='pathmnist',
        data_dir=os.path.join(LOCAL_DATA, 'pathmnist'),
        download=True,  # Auto-download from Zenodo
    )
    train_ds, test_ds = loader.load()
    print("✅ PathMNIST loaded successfully!")
    
    # Validate and visualize
    loader.validate_and_visualize(
        num_samples=5,
        output_dir=validation_output,
    )
    print(f"\n✅ Validation complete! Sample images saved to {validation_output}")
    
except Exception as e:
    print(f"❌ Error validating PathMNIST: {e}")
    raise

In [ ]:
!python scripts/run_sweep.py --experiment pout_sweep_pathmnist --resume

In [ ]:
!python scripts/run_analysis.py --results_dir outputs/pout_sweep_pathmnist --analysis surface
!python scripts/run_visualization.py --results_dir outputs/pout_sweep_pathmnist